# 🕵️‍♂️ FASE 5: Explainable AI (Interpretasi SHAP pada LSTM)
## Prediksi PM2.5 di Cekungan Bandung
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini adalah puncak pembuktian riset Anda. Model Deep Learning (LSTM) terkenal sebagai "Kotak Hitam" (Black Box) karena kita tidak tahu persis bagaimana miliaran perhitungan di dalamnya menghasilkan angka prediksi.

Dengan menggunakan metode **SHAP (SHapley Additive exPlanations)** yang memenangkan penghargaan Nobel Teori Permainan (Game Theory), kita akan membongkar isi otak LSTM dan mengungkap **Faktor Cuaca Apa yang Paling Memicu Lonjakan Polusi PM2.5** di Bandung.


## 1. Import Library
Pastikan Anda sudah menginstall library `shap`. Jika belum, jalankan `pip install shap` di terminal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import shap
import tensorflow as tf
from tensorflow.keras.models import load_model

# Matikan warning keras/tf yang mengganggu
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print(f"[OK] Library loaded. Versi SHAP: {shap.__version__}")


## 2. Load Data, Model LSTM & Transformasi 3D
Sama seperti Fase 4, kita harus mengonversi data menjadi 3D Tensor sebelum memasukkannya ke dalam model dan mesin SHAP.


In [ ]:
DATA_DIR = r'D:\Kuliah Praktik\KP BRIN\data\processed'
MODEL_DIR = r'D:\Kuliah Praktik\KP BRIN\models'

# Load Model LSTM
model = load_model(os.path.join(MODEL_DIR, 'lstm_model.keras'))
print("[OK] Model LSTM berhasil dimuat.")

# Load Data Processed
train_scaled = pd.read_csv(os.path.join(DATA_DIR, 'train_scaled.csv'), index_col=0, parse_dates=True)
test_scaled = pd.read_csv(os.path.join(DATA_DIR, 'test_scaled.csv'), index_col=0, parse_dates=True)

target_col = 'pm25'
feature_names = [c for c in train_scaled.columns if c != target_col]

# Fungsi Transformasi 3D (dari Fase 4)
def create_sequences(df, target_col, time_steps=24):
    X, y = [], []
    data_array = df.values
    target_idx = df.columns.get_loc(target_col)
    for i in range(len(data_array) - time_steps):
        X.append(data_array[i:(i + time_steps)])
        y.append(data_array[i + time_steps, target_idx])
    return np.array(X), np.array(y)

TIME_STEPS = 24

# Kita hanya butuh data X (Fitur) untuk SHAP
X_train_seq, _ = create_sequences(train_scaled, target_col, TIME_STEPS)
X_test_seq, _ = create_sequences(test_scaled, target_col, TIME_STEPS)

print(f"Tensor Input SHAP: {X_test_seq.shape}")


## 3. Komputasi SHAP (KernelExplainer + Wrapper 2D)
Karena pembaruan versi TensorFlow terbaru (TF 2.15+ / Keras 3), `GradientExplainer` dan `DeepExplainer` mengalami *bug* internal (`get_session` error). 
Sebagai solusinya, kita menggunakan pendekatan yang jauh lebih stabil yaitu **Model-Agnostic KernelExplainer**.

Karena KernelExplainer hanya menerima data 2D (tabel biasa), kita membuat fungsi *wrapper* untuk mengubah data 2D menjadi 3D secara transparan saat memanggil model.


In [ ]:
print("Memulai komputasi SHAP menggunakan KernelExplainer...")

# 1. Background Sample (Sebagai referensi titik "nol" bagi SHAP)
np.random.seed(42)
background_idx = np.random.choice(X_train_seq.shape[0], 50, replace=False) # Kita kurangi jadi 50 agar komputasi tidak terlalu lama
background = X_train_seq[background_idx]

# 2. Test Sample (Data yang akan dianalisis)
test_idx = np.random.choice(X_test_seq.shape[0], 50, replace=False)
test_sample = X_test_seq[test_idx]

# --- FUNGSI WRAPPER 3D ke 2D ---
# Fitur asli kita adalah (24 jam x 15 variabel) = 360 kolom datar.
def lstm_predict_wrapper(x_2d_flat):
    # Kembalikan ke bentuk 3D: [Jumlah Sampel, 24 Jam, 15 Fitur]
    x_3d = x_2d_flat.reshape(-1, TIME_STEPS, X_train_seq.shape[2])
    # KernelExplainer mengharapkan output 1D array (N,) untuk regresi tunggal
    return model.predict(x_3d, verbose=0).flatten()

# Pipihkan (Flatten) data dari 3D ke 2D untuk KernelExplainer
background_2d = background.reshape(background.shape[0], -1)
test_sample_2d = test_sample.reshape(test_sample.shape[0], -1)

# 3. Inisialisasi KernelExplainer
# Catatan: Akan memunculkan progress bar. Bisa memakan waktu 3-5 menit karena harus mencoba ribuan kombinasi (Game Theory).
explainer = shap.KernelExplainer(lstm_predict_wrapper, background_2d)

# 4. HITUNG SHAP VALUES!
shap_values = explainer.shap_values(test_sample_2d)

# shap_values berbentuk [50, 360]. Kita kembalikan ke 3D [50, 24, 15]
shap_values_3d = shap_values.reshape(-1, TIME_STEPS, X_train_seq.shape[2])

print(f"\n[SELESAI] Bentuk matriks SHAP 3D yang berhasil diekstrak: {shap_values_3d.shape}")


## 4. Flattening (Meratakan Waktu)
Hasil SHAP kita berbentuk 3D `(200 sampel, 24 jam ke belakang, 14 fitur)`.
Visualisasi 3D sangat sulit dibaca di kertas laporan. Oleh karena itu, kita akan **merata-ratakan (Mean)** efek di sepanjang 24 jam tersebut agar menjadi 2D `(200 sampel, 14 fitur)`.
Dengan begitu, kita bisa menjawab pertanyaan: *"Secara rata-rata dalam 24 jam terakhir, fitur mana yang paling penting?"*


In [ ]:
# Rata-ratakan pada sumbu 1 (Sumbu Time Steps/24 jam)
shap_values_2d = np.mean(shap_values_3d, axis=1)

# Lakukan hal yang sama pada data aslinya (Test Sample)
test_sample_2d = np.mean(test_sample, axis=1)

print(f"Bentuk SHAP sekarang (Siap di-plot): {shap_values_2d.shape}")


## 5. Visualisasi 1: SHAP Summary Plot (Global Importance)
Grafik Beeswarm ini adalah visualisasi terpenting untuk Bab Hasil Anda.
- **Sumbu Y:** Daftar fitur dari yang terpenting (atas) ke paling tidak penting (bawah).
- **Sumbu X:** Nilai SHAP (Ke kanan berarti mendorong Polusi NAIK, ke kiri mendorong TURUN).
- **Warna:** Merah berarti nilai fitur aslinya Tinggi (misal Suhu panas), Biru berarti Rendah.


In [ ]:
plt.figure(figsize=(10, 8))
plt.title("SHAP Summary Plot: Faktor Pendorong PM2.5 di Cekungan Bandung\n", fontsize=14, fontweight='bold')

shap.summary_plot(
    shap_values_2d, 
    test_sample_2d, 
    feature_names=feature_names,
    plot_type="dot",
    show=False
)
plt.tight_layout()
plt.show()


## 6. Visualisasi 2: Hubungan Curah Hujan/Suhu terhadap Polusi
Grafik ini menyoroti satu variabel tertentu secara mendalam.
Contoh: Kita ingin melihat batas di mana Suhu (Suhu 2m C) mulai memicu lonjakan PM2.5.


In [ ]:
# Cari indeks kolom suhu
target_feature = 'suhu_2m_C'
feature_idx = feature_names.index(target_feature)

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    feature_idx, 
    shap_values_2d, 
    test_sample_2d, 
    feature_names=feature_names,
    interaction_index=None, # Matikan interaksi warna agar tidak membingungkan
    title=f"Hubungan Non-Linear: {target_feature} vs Kenaikan PM2.5",
    show=False
)
plt.grid(True, alpha=0.3)
plt.show()
